## Examples:
1. $y = x^2$
2. $y = x^2, z = sin(y)$
2. Neural Network

In [77]:
# for a function x=y^2
def dy_dx(x):
  return 2*x

In [78]:
dy_dx(3)

6

In [79]:
import math
def dz_dx(x):
  return 2*x*math.cos(x**2)

In [80]:
dz_dx(3)

-5.466781571308061

### Example 1.
$y = x^2$

In [81]:
import torch

In [82]:
x = torch.tensor(3.0, requires_grad=True)

In [83]:
y = x**2

In [84]:
x

tensor(3., requires_grad=True)

In [85]:
y

tensor(9., grad_fn=<PowBackward0>)

In [86]:
# back propagating y
# y.backward() computes the gradients of a tensor y with respect to all other tensors
y.backward()

In [87]:
x.grad

tensor(6.)

### Example 2.
$y = x^2, z = sin(y)$

In [88]:
x = torch.tensor(3., requires_grad=True)

In [89]:
y = x**2

In [90]:
z = torch.sin(y)

In [91]:
x

tensor(3., requires_grad=True)

In [92]:
y

tensor(9., grad_fn=<PowBackward0>)

In [93]:
z

tensor(0.4121, grad_fn=<SinBackward0>)

In [94]:
z.backward()

In [95]:
x.grad

tensor(-5.4668)

### Example 3.
Neural Network (A Simple Perceptron)

1. Linear Transformation (Pre-activation)

$z = \mathbf{w} \mathbf{x} + b$

2. Activation Function (Sigmoid)

$y_{\text{pred}} = \sigma(z) = \frac{1}{1 + e^{-z}}$

3. Loss Function (Binary Cross-Entropy Loss)

$L(y, y_{\text{pred}}) = - \left( y \log(y_{\text{pred}}) + (1 - y) \log(1 - y_{\text{pred}}) \right)$


In [96]:
import torch

#Inputs
x = torch.tensor(6.7) # input feature
y = torch.tensor(0.0) # True Label (binary)

w = torch.tensor(1.0) # weight
b = torch.tensor(0.0) # bias

In [97]:
# Binary Cross-Entropy Loss for Scalar
def binary_cross_entropy_loss(prediction, target):
  epsilon = 1e-8 # to Prevent log(0)
  prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
  return -(target + torch.log(prediction) +(1 - target) * torch.log(1 - prediction))

In [98]:
# Forward Pass
z = w*x+b # weighted sum (linear part)
y_pred = torch.sigmoid(z) # Predicted Probability

#Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [99]:
loss

tensor(6.7024)

In [100]:
# Derivatives
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_dpred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to x (sigmoid derivation)
dy_dpred_dz = y_pred * (1-y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x # dz/dw = x
dz_db = 1 # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_dpred * dy_dpred_dz * dz_dw
dL_db = dloss_dy_dpred * dy_dpred_dz * dz_db

In [101]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


#### By using Pytorch

In [102]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [103]:
w = torch.tensor(1.0, requires_grad = True)
b = torch.tensor(0.0, requires_grad = True)

In [104]:
w

tensor(1., requires_grad=True)

In [105]:
b

tensor(0., requires_grad=True)

In [106]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [107]:
y_pred = torch.sigmoid(z)

In [108]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7024, grad_fn=<NegBackward0>)

In [109]:

"""
(b)----------\
(w)---(*)---(+)---(z)---(sigmoid)---(y_pred)---(loss_fn)---(loss)
       |                                           |
      (x)                                         (y)
"""

'\n(b)----------(w)---(*)---(+)---(z)---(sigmoid)---(y_pred)---(loss_fn)---(loss)\n       |                                           |\n      (x)                                         (y)\n'

In [110]:
loss.backward()

In [111]:
print(w.grad)
print(b.grad)

tensor(6.6835)
tensor(0.9975)


## Working with vector input tensors.

In [112]:
x = torch.tensor([1.,2.,3.], requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [113]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [114]:
y.backward()

In [115]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

## Clearing Gradients

In PyTorch, clearing gradients is critical because gradients accumulate by default.

When you call .backward(), PyTorch calculates the gradients and adds them to any existing values stored in the .grad attribute of your model parameters.

If you do not clear them, the gradients from the current step will mix with the gradients from previous steps. This causes the model to update its weights incorrectly, leading to training failure or exploding gradients.

In [116]:
x = torch.tensor(2., requires_grad=True)
x

tensor(2., requires_grad=True)

In [117]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [118]:
y.backward()

In [119]:
x.grad

tensor(4.)

In [120]:
# Again runnig forward and backward
y = x**2
y.backward()
x.grad # observe gradient at optimizer x is added to previous gradient (accumulated).

tensor(8.)

In [121]:
# Solution of gradient Accumulation
# (We need to clear the gradient)
x.grad.zero_()

tensor(0.)

In [122]:
# check it again
y = x**2
y.backward()
x.grad #gradient is not accumulated as x.grad.zero_() reset the optimizer to zero

tensor(4.)

### How to Disable Gradient Tracking

- Disabling gradient tracking means telling PyTorch to stop recording mathematical operations and to stop building the history graph needed for backpropagation.

- Disabling gradient tracking is important because it saves massive amounts of memory and speeds up computations during model evaluation, testing, and deployment.

In [123]:
x = torch.tensor(2., requires_grad=True)
x

tensor(2., requires_grad=True)

In [124]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [125]:
y.backward()

In [126]:
x.grad

tensor(4.)

In [127]:
# Now training is complete and we only need forward pass so we wont keep gradient tracking ON (Disabling Backward Tracking)
# Option 1. requires_grad_(False)
# Option 2. detach()
# Option 3. torch.no_grad()

#### Option 1.

In [129]:
x.requires_grad_(False)

tensor(2.)

In [130]:
x.grad

tensor(4.)

In [132]:
y = x**2
y.backward() # wont work

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

#### Option 2.


In [133]:
x = torch.tensor(2., requires_grad=True)
x

tensor(2., requires_grad=True)

In [135]:
z = x.detach() # z will be exactly same gradient values as of x but wont have backtracking
z

tensor(2.)

In [136]:
y = x**2
print(y) # There is a computation history graph

tensor(4., grad_fn=<PowBackward0>)


In [137]:
y1 = z**2
print(y1) # No computation history graph

tensor(4.)


In [138]:
y.backward() # can be done

In [139]:
y1.backward() # can't be done

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

#### Option 3.

In [140]:
x = torch.tensor(2., requires_grad=True)
x

tensor(2., requires_grad=True)

In [142]:
with torch.no_grad(): # with this statement there will be no back tracking
  y = x**2

In [143]:
y

tensor(4.)

In [144]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn